# Objective

This notebook follows "Script_mrc_to_h5_datacube.ipynb". The data has been reshaped, shifted and turned into a py4DSTEM 'Datacube'. The next step is to do some rudamentary analysis:

- Plot the average/max diffraction pattern across the full dataset.
- Create a brightfield image to see the nanowire's shape. It should match the image taken using HAADF.
- Extract the vacuum probe (the G=0 peak observed in a region where there is no nanowire crystal obscuring the electron beam).
- Create a basic interactive GUI to click through the pixels in the brightfield to pull up their diffraction patterns. We'll use this to extract various diffraction patterns which correspond to interesting crystal phases. These 'target' diffraction patterns will be key to later analysis.

In [ ]:
import py4DSTEM
import numpy as np
import matplotlib.pyplot as plt
from py4DSTEM import show

py4DSTEM.__version__

In [ ]:
# plt.rcParams['figure.dpi'] = 200  # or any DPI value you want

In [ ]:
plt.rcParams.update({
    'axes.titlesize': 15,        # Title of the axes
    'axes.labelsize': 15,        # Axis labels
    'xtick.labelsize': 15,       # X-axis tick labels
    'ytick.labelsize': 15,       # Y-axis tick labels
    'legend.fontsize': 15,       # Legend text
    'figure.titlesize': 15       # Overall figure title (if using suptitle)
})

In [ ]:
data_path = 'file.h5'


In [ ]:
# Load data

datacube = py4DSTEM.read(data_path)

In [ ]:
# Mean diffraction
dp_mean = datacube.get_dp_mean()
show(
    dp_mean,
    scaling='log',
    show_cbar = True
)

In [ ]:
# Max diffraction
dp_max = datacube.get_dp_max()
show(
    dp_max,
    scaling='log',
    show_cbar = True
)

In [ ]:
datacube.tree()

# Bright-Field

In [ ]:
# Get the probe position and size
probe_semiangle, probe_qx0, probe_qy0 = datacube.get_probe_size(dp_max.data)
print('Estimated probe radius =', '%.2f' % probe_semiangle, 'pixels')

# Set detector geometry
center = probe_qx0, probe_qy0
radius = probe_semiangle * 7

# overlay selected detector position over max dp (default)
datacube.position_detector(
    mode = 'circle',
    geometry = (center, radius))

In [ ]:
# Capture the virtual BF
datacube.get_virtual_image(
    mode = 'circle',
    geometry = (center,radius),
    name = 'bright_field', 
)

show( datacube.tree('bright_field') )

In [ ]:
np.save('brightfield.npy', datacube.tree('bright_field').data, allow_pickle=True)

The bright areas are where there is no diffraction- we can use those to extract the vacuum probe for disk detection.

In [ ]:
import numpy as np
mask = np.zeros(datacube.Rshape,dtype=bool)
# Set mask region (below) to an area outside the nanowire. 
mask[105:110,60:65] = 1

show(
    datacube.tree('bright_field'),
    mask = ~mask,
    mask_alpha = 0.667,
    mask_color = 'r'
)

In [ ]:
# generate a probe
%matplotlib notebook

probe = datacube.get_vacuum_probe( ROI=mask )
alpha_pr,qx0_pr,qy0_pr = py4DSTEM.process.calibration.get_probe_size( probe.probe )

show(probe.probe, vmin=-1, vmax=1, show_cbar = True)

**NOTE:** In the above plot, I used the "zoom" functionality in the Matplotlib interactive plot.

In [ ]:
# # prepare the probe kernel
# probe.get_kernel(mode='flat')

# show(probe.kernel)

# Interactive GUI

In [ ]:
%matplotlib inline
plt.imshow(datacube.tree('bright_field').data, cmap='grey')
plt.show()

In [ ]:
def gui(datacube):
    brightfield = datacube.tree('bright_field').data
    data4d      = datacube.data
    m, n = brightfield.shape   
    
    # Plot the brightfield image. It will never be changed/updated.
    fig, (ax_img, ax_dp) = plt.subplots(1, 2, figsize=(10, 4))
    im = ax_img.imshow(brightfield, cmap='grey')
    ax_img.set_title("Brightfield (click a pixel)")
    dp_plot = ax_dp.imshow(np.zeros(data4d.shape[2:]), cmap='gray')
    ax_dp.set_title("Click a pixel → diffraction here")
    
    selected_pixel = None # Stores the Artist (dot) on the pixel corresponding to plotted diffraction pattern.
    
    def onclick(event):
        nonlocal dp_plot, selected_pixel
        if event.inaxes != ax_img:
            return
        mpl_x = int(round(event.xdata))
        mpl_y = int(round(event.ydata))
        
        # Remember, the 2D 'imshow' plot has rows on y-axis and columns on x-axis. So, we
        # must index as data[y, x].
        np_x = mpl_y
        np_y = mpl_x
        if 0 <= np_x < m and 0 <= np_y < n:
            # Extract and plot the diffraction pattern selected from datacube.
            dp = data4d[np_x, np_y]
            dp_log = np.log(dp)
            dp_plot.set_data(dp_log)
            dp_plot.set_clim(vmin=dp_log.min(), vmax=dp_log.max())
            ax_dp.set_title(f"Diffraction at index [{np_x}, {np_y}]")
            
            # Plot a dot on the selected pixel of the darkfield/boolean map.
            if selected_pixel:
                selected_pixel.remove()
            selected_pixel = ax_img.scatter([np_y], [np_x], s=1, facecolors='red')
            
            # Update the canvas.
            fig.canvas.draw_idle()
    
    fig.canvas.mpl_connect('button_press_event', onclick)
    plt.tight_layout()
    plt.show()


# Looking for interesting crystal phases

I ran the gui several times to find diffraction patterns which correspond to different crystal phases. Below, we see three distinct patterns which could correspond to lead, zincblende and wurtzite.

In [ ]:
%matplotlib notebook
gui(datacube)

In [ ]:
gui(datacube)

In [ ]:
gui(datacube)